# 💳 Automated ML Benchmarking — UCI Credit Card Default
---
**Dataset:** UCI Credit Card Default (`UCI_Credit_Card.csv`)  
**Tujuan:** Memprediksi apakah nasabah akan **gagal bayar (default)** bulan depan  
**Pendekatan:** Benchmarking komparasi skala besar (28 model) menggunakan arsitektur `Pipeline` scikit-learn

---
## Daftar Isi
1. [Instalasi & Import Library](#1)
2. [Load & Eksplorasi Data](#2)
3. [Analisis Demografi vs Risiko Default](#3)
4. [Identifikasi Fitur Prediktor Terkuat](#4)
5. [Definisi Fitur & Arsitektur Pipeline](#5)
6. [Benchmarking Semua Model](#6)
7. [Visualisasi Hasil Benchmarking](#7)
8. [Simpan & Load Model (Pickle)](#8)
9. [Prediksi Nasabah Baru](#9)
10. [Kesimpulan](#10)


## 1. Instalasi & Import Library <a id='1'></a>

In [ ]:
# ── Instalasi paket eksternal (jalankan sekali di Colab) ──────────────────────
!pip install xgboost lightgbm catboost --quiet


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pickle, os, io, time, warnings
warnings.filterwarnings('ignore')

# ── Sklearn Core ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve)

# ── Model Lengkap ─────────────────────────────────────────────────────────────
from sklearn.linear_model import (LogisticRegression, SGDClassifier,
                                   RidgeClassifier, PassiveAggressiveClassifier)
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, BaggingClassifier,
                               ExtraTreesClassifier, HistGradientBoostingClassifier)
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                            QuadraticDiscriminantAnalysis)
from sklearn.neural_network import MLPClassifier

from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ── Style matplotlib ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F9FA',
    'axes.edgecolor':   '#DEE2E6',
    'axes.grid':        True,
    'grid.color':       '#E9ECEF',
    'grid.linestyle':   '--',
    'grid.alpha':       0.7,
    'font.family':      'DejaVu Sans',
})

MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("✅ Semua library berhasil diimport")


## 2. Load & Eksplorasi Data <a id='2'></a>

### 2.1 Upload Dataset ke Google Colab

Jalankan cell di bawah untuk mengupload file `UCI_Credit_Card.csv` dari komputer lokal ke Colab.  
Atau jika file sudah ada di Google Drive, mount Drive terlebih dahulu (tersedia di cell alternatif).


In [ ]:
# ── Opsi A: Upload langsung dari komputer lokal ───────────────────────────────
from google.colab import files

print("📂 Silakan pilih file UCI_Credit_Card.csv dari komputer kamu...")
uploaded = files.upload()

CSV_PATH = list(uploaded.keys())[0]
print(f"✅ File '{CSV_PATH}' berhasil diupload")


In [ ]:
# ── Opsi B: Mount Google Drive (uncomment jika pakai Drive) ──────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/UCI_Credit_Card.csv'


In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)

print(f"{'='*50}")
print(f"  INFORMASI DATASET")
print(f"{'='*50}")
print(f"  Total baris    : {df.shape[0]:,}")
print(f"  Total kolom    : {df.shape[1]}")
print(f"  Missing values : {df.isnull().sum().sum()}")
distribusi = df['default.payment.next.month'].value_counts()
print(f"  Kelas 0 (Lancar)     : {distribusi[0]:,}  ({distribusi[0]/len(df)*100:.1f}%)")
print(f"  Kelas 1 (Default)    : {distribusi[1]:,}  ({distribusi[1]/len(df)*100:.1f}%)")
print(f"  Rasio imbalance      : 1 : {distribusi[0]//distribusi[1]}")
print(f"{'='*50}")


In [ ]:
# ── Preview data ──────────────────────────────────────────────────────────────
print("=== 5 Baris Pertama ===")
df.head()


In [ ]:
# ── Statistik deskriptif ──────────────────────────────────────────────────────
print("=== Statistik Deskriptif ===")
df.describe().round(2)


In [ ]:
# ── Keterangan kolom ─────────────────────────────────────────────────────────
metadata = {
    'ID'            : 'Identifikasi unik setiap nasabah',
    'LIMIT_BAL'     : 'Jumlah kredit yang diberikan (NT Dollar)',
    'SEX'           : 'Jenis Kelamin (1=Laki-laki, 2=Perempuan)',
    'EDUCATION'     : 'Tingkat Pendidikan (1=S2/S3, 2=S1, 3=SMA, 4=Lainnya)',
    'MARRIAGE'      : 'Status Pernikahan (1=Menikah, 2=Lajang, 3=Lainnya)',
    'AGE'           : 'Usia nasabah (Tahun)',
    'PAY_0..PAY_6'  : 'Status pembayaran bulanan Sep–Apr 2005 (-1=tepat, 1=terlambat 1bln, dst)',
    'BILL_AMT1..6'  : 'Jumlah tagihan bulanan Sep–Apr 2005',
    'PAY_AMT1..6'   : 'Jumlah pembayaran nominal bulanan Sep–Apr 2005',
    'default.payment.next.month' : 'TARGET: 1=Gagal Bayar, 0=Lancar',
}
print("=== Keterangan Kolom Dataset ===")
for k, v in metadata.items():
    print(f"  {k:<30} → {v}")


## 3. Analisis Demografi vs Risiko Default <a id='3'></a>

> 💡 **Pertanyaan Eksplorasi:** *How does the probability of default payment vary
> by categories of different demographic variables?*


In [ ]:
mapping_labels = {
    "SEX":       {1: "1. Male", 2: "2. Female"},
    "EDUCATION": {1: "1. Graduate School", 2: "2. University",
                  3: "3. High School",     4: "4. Others",
                  5: "5. Unknown",         6: "6. Unknown"},
    "MARRIAGE":  {1: "1. Married", 2: "2. Single", 3: "3. Others"},
}
demo_vars = ["SEX", "EDUCATION", "MARRIAGE"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, var in zip(axes, demo_vars):
    df_tmp = df.copy()
    df_tmp[var] = df_tmp[var].map(mapping_labels[var])

    grouped = (df_tmp.groupby(var)['default.payment.next.month']
               .mean().reset_index())
    grouped['default.payment.next.month'] *= 100
    grouped = grouped.sort_values('default.payment.next.month', ascending=False)

    bars = ax.bar(grouped[var], grouped['default.payment.next.month'],
                  color=sns.color_palette('viridis', len(grouped)),
                  edgecolor='white', linewidth=1.2)

    for bar, val in zip(bars, grouped['default.payment.next.month']):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.3,
                f"{val:.1f}%", ha='center', va='bottom',
                fontsize=10, fontweight='bold')

    ax.set_title(f"Risiko Default per {var}", fontsize=13, pad=10)
    ax.set_ylabel("Rasio Gagal Bayar (%)")
    ax.set_xlabel(f"Kategori {var}")
    ax.tick_params(axis='x', rotation=15)

plt.suptitle("Probabilitas Gagal Bayar Berdasarkan Variabel Demografi",
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("eda_demografi.png", dpi=150, bbox_inches='tight')
plt.show()


## 4. Identifikasi Fitur Prediktor Terkuat <a id='4'></a>

> 💡 **Pertanyaan Eksplorasi:** *Which variables are the strongest predictors of default payment?*


In [ ]:
# ── Korelasi terhadap target ──────────────────────────────────────────────────
corr = (df.corr()['default.payment.next.month']
        .drop(['ID', 'default.payment.next.month'])
        .reset_index())
corr.columns = ['Fitur', 'Korelasi']
corr['Abs'] = corr['Korelasi'].abs()
corr = corr.sort_values('Abs', ascending=False).reset_index(drop=True)

print("=== Top 10 Fitur Paling Berpengaruh ===")
print(corr.head(10)[['Fitur', 'Korelasi']].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart korelasi
colors = ['#E74C3C' if v > 0 else '#3498DB' for v in corr['Korelasi']]
axes[0].barh(corr['Fitur'], corr['Korelasi'], color=colors, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title("Korelasi Semua Fitur terhadap Target Default", fontsize=13)
axes[0].set_xlabel("Koefisien Korelasi")
axes[0].set_ylabel("Nama Fitur")
axes[0].invert_yaxis()

# Heatmap top fitur
top_cols  = corr.head(10)['Fitur'].tolist() + ['default.payment.next.month']
heatmap_data = df[top_cols].corr()
mask = np.triu(np.ones_like(heatmap_data, dtype=bool))
sns.heatmap(heatmap_data, ax=axes[1], mask=mask, annot=True, fmt=".2f",
            cmap='RdBu_r', center=0, linewidths=0.5,
            annot_kws={'size': 8})
axes[1].set_title("Heatmap Korelasi Antar Top-10 Fitur", fontsize=13)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("eda_korelasi.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Distribusi target ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = ['Lancar (0)', 'Default (1)']
sizes  = [distribusi[0], distribusi[1]]
colors = ['#2ECC71', '#E74C3C']

axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.2f%%',
            startangle=90, explode=[0, 0.07],
            textprops={'fontsize': 12},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title("Distribusi Kelas Target", fontsize=13)

axes[1].bar(labels, sizes, color=colors, edgecolor='white', width=0.4)
for i, v in enumerate(sizes):
    axes[1].text(i, v + 200, f"{v:,}", ha='center', fontsize=12, fontweight='bold')
axes[1].set_title("Jumlah Sampel per Kelas", fontsize=13)
axes[1].set_ylabel("Jumlah Nasabah")

plt.suptitle("Class Imbalance — Dataset UCI Credit Card",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("eda_distribusi_target.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"⚠️  Dataset imbalanced: kelas Lancar {distribusi[0]/len(df)*100:.1f}% vs Default {distribusi[1]/len(df)*100:.1f}%")
print("   → Model akan menggunakan class_weight='balanced' untuk menangani ini")


## 5. Definisi Fitur & Arsitektur Pipeline <a id='5'></a>

Dataset memiliki dua jenis fitur yang membutuhkan perlakuan berbeda:

| Jenis Fitur | Kolom | Perlakuan |
|---|---|---|
| **Numerik Kontinu** | LIMIT_BAL, AGE, BILL_AMT*, PAY_AMT* | `StandardScaler` — normalisasi skala besar |
| **Kategorikal / Status** | SEX, EDUCATION, MARRIAGE, PAY_0..PAY_6 | `passthrough` — diteruskan apa adanya |

Keduanya digabung dalam `ColumnTransformer`, lalu dirangkai ke classifier via `Pipeline`
sehingga **preprocessing dan model menjadi satu objek yang bisa disimpan sebagai pickle**.


In [ ]:
# ── Definisi kolom ────────────────────────────────────────────────────────────
kolom_numerik_kontinu = [
    'LIMIT_BAL', 'AGE',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1',  'PAY_AMT2',  'PAY_AMT3',  'PAY_AMT4',  'PAY_AMT5',  'PAY_AMT6'
]

kolom_kategorikal_status = [
    'SEX', 'EDUCATION', 'MARRIAGE',
    'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'
]

TARGET = 'default.payment.next.month'

# ── Split fitur & target ──────────────────────────────────────────────────────
X = df.drop(columns=['ID', TARGET])
y = df[TARGET]

# ── Train-Test Split (stratified) ─────────────────────────────────────────────
TEST_SIZE    = 0.20
RANDOM_STATE = 10

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"✅ Data berhasil dipisah")
print(f"   X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"   X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")
print(f"   Proporsi default di train : {y_train.mean()*100:.2f}%")
print(f"   Proporsi default di test  : {y_test.mean()*100:.2f}%")


In [ ]:
# ── Preprocessor (ColumnTransformer) ─────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num_scale', StandardScaler(), kolom_numerik_kontinu),
        ('cat_keep', 'passthrough',     kolom_kategorikal_status)
    ]
)

# Contoh cara membuat pipeline untuk satu model:
# pipeline = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('classifier',   RandomForestClassifier())
# ])

print("✅ Preprocessor berhasil didefinisikan")
print(f"   Kolom numerik   ({len(kolom_numerik_kontinu)}) : {kolom_numerik_kontinu}")
print(f"   Kolom kategorial ({len(kolom_kategorikal_status)}) : {kolom_kategorikal_status}")


## 6. Benchmarking Semua Model <a id='6'></a>

Kita akan melatih dan mengevaluasi **28 model** secara otomatis menggunakan
arsitektur Pipeline. Setiap model diukur dengan metrik:

| Metrik | Penjelasan |
|---|---|
| **Accuracy** | Proporsi prediksi yang benar secara keseluruhan |
| **Precision** | Dari semua yang diprediksi Default, berapa % yang benar? |
| **Recall** | Dari semua nasabah yang benar-benar Default, berapa % yang terdeteksi? |
| **F1-Score** | Harmonic mean Precision & Recall |
| **ROC-AUC** | Kemampuan model membedakan kelas (0–1, semakin tinggi semakin baik) |
| **Durasi** | Waktu training (detik) |

Model yang gagal (error) akan dicatat dan dilewati otomatis.


In [ ]:
# ── Definisi semua model ──────────────────────────────────────────────────────
rs = RANDOM_STATE

all_models = {
    # Linear
    "Logistic Regression"         : LogisticRegression(class_weight='balanced', solver='liblinear', random_state=rs),
    "SGD Classifier"              : SGDClassifier(class_weight='balanced', random_state=rs, loss='log_loss'),
    "Ridge Classifier"            : RidgeClassifier(class_weight='balanced', random_state=rs),
    "Passive Aggressive"          : PassiveAggressiveClassifier(class_weight='balanced', random_state=rs, max_iter=1000),
    # Naive Bayes
    "Gaussian NB"                 : GaussianNB(),
    "Bernoulli NB"                : BernoulliNB(),
    # SVM
    "Linear SVC"                  : LinearSVC(class_weight='balanced', random_state=rs, dual=False),
    # Tree-based
    "Decision Tree"               : DecisionTreeClassifier(random_state=rs),
    "Extra Tree (single)"         : ExtraTreeClassifier(random_state=rs),
    # Ensemble
    "Random Forest"               : RandomForestClassifier(class_weight='balanced', random_state=rs),
    "Extra Trees"                 : ExtraTreesClassifier(random_state=rs),
    "Gradient Boosting"           : GradientBoostingClassifier(random_state=rs),
    "Hist Gradient Boosting"      : HistGradientBoostingClassifier(random_state=rs),
    "AdaBoost"                    : AdaBoostClassifier(random_state=rs),
    "Bagging"                     : BaggingClassifier(random_state=rs),
    # Neighbors & Centroid
    "K-Nearest Neighbors"         : KNeighborsClassifier(),
    "Nearest Centroid"            : NearestCentroid(),
    # Discriminant Analysis
    "LDA"                         : LinearDiscriminantAnalysis(),
    "QDA"                         : QuadraticDiscriminantAnalysis(),
    # Neural Network
    "MLP Neural Network"          : MLPClassifier(random_state=rs, max_iter=500),
    # External Boosting
    "XGBoost"                     : XGBClassifier(random_state=rs, eval_metric='logloss', verbosity=0),
    "LightGBM"                    : LGBMClassifier(random_state=rs, is_unbalance=True, verbose=-1),
    "CatBoost"                    : CatBoostClassifier(verbose=0, random_state=rs, auto_class_weights='Balanced'),
}

print(f"✅ Total model yang akan di-benchmark: {len(all_models)}")
for i, name in enumerate(all_models, 1):
    print(f"   {i:>2}. {name}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BENCHMARKING LOOP
# Setiap model dilatih, dievaluasi, dan hasilnya disimpan.
# Model yang error otomatis dilewati dengan keterangan alasan.
# ══════════════════════════════════════════════════════════════════════════════

results        = []
trained_pipelines = {}   # Simpan pipeline terlatih untuk pickle nanti

print("=" * 70)
print(f"  MEMULAI BENCHMARKING — {len(all_models)} MODEL")
print("=" * 70)

for idx, (name, model_obj) in enumerate(all_models.items(), 1):
    print(f"[{idx:>2}/{len(all_models)}] {name:<35}", end=" ", flush=True)

    try:
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier',   model_obj)
        ])

        t0 = time.time()
        pipeline.fit(X_train, y_train)
        durasi = time.time() - t0

        y_pred = pipeline.predict(X_test)

        # Probabilitas / decision_function untuk ROC-AUC
        if hasattr(pipeline.named_steps['classifier'], "predict_proba"):
            y_scores = pipeline.predict_proba(X_test)[:, 1]
        elif hasattr(pipeline.named_steps['classifier'], "decision_function"):
            y_scores = pipeline.decision_function(X_test)
        else:
            y_scores = None

        acc  = accuracy_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        roc  = roc_auc_score(y_test, y_scores) if y_scores is not None else np.nan

        results.append({
            'Model'    : name,
            'Accuracy' : acc,
            'Precision': prec,
            'Recall'   : rec,
            'F1-Score' : f1,
            'ROC-AUC'  : roc,
            'Durasi(s)': round(durasi, 2),
            'Status'   : '✅ OK'
        })
        trained_pipelines[name] = pipeline
        print(f"ROC={roc:.4f}  F1={f1:.4f}  [{durasi:.1f}s] ✅")

    except Exception as e:
        alasan = str(e).split('\n')[0][:60]
        results.append({
            'Model'    : name,
            'Accuracy' : np.nan, 'Precision': np.nan,
            'Recall'   : np.nan, 'F1-Score' : np.nan,
            'ROC-AUC'  : np.nan, 'Durasi(s)': 0,
            'Status'   : f'❌ {alasan}'
        })
        print(f"❌ GAGAL — {alasan}")

print()
print("=" * 70)
print(f"  SELESAI — {len(trained_pipelines)} model berhasil, {len(results)-len(trained_pipelines)} gagal")
print("=" * 70)


In [ ]:
# ── Leaderboard ───────────────────────────────────────────────────────────────
results_df = (pd.DataFrame(results)
              .sort_values('ROC-AUC', ascending=False)
              .reset_index(drop=True))
results_df.insert(0, 'Rank', results_df.index + 1)

# Format untuk display
disp = results_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    disp[col] = disp[col].apply(lambda x: f"{x*100:.2f}%" if not pd.isna(x) else "N/A")

print("\n📋 LEADERBOARD BENCHMARKING (Diurutkan ROC-AUC)")
print(disp[['Rank','Model','ROC-AUC','F1-Score','Accuracy','Recall','Precision','Durasi(s)','Status']].to_string(index=False))


In [ ]:
# ── Identifikasi pemenang ─────────────────────────────────────────────────────
best_row  = results_df.dropna(subset=['ROC-AUC']).iloc[0]
best_name = best_row['Model']
best_auc  = best_row['ROC-AUC']
best_f1   = best_row['F1-Score']

print(f"\n{'='*55}")
print(f"  🏆 MODEL TERBAIK: {best_name}")
print(f"     ROC-AUC  : {best_auc*100:.2f}%")
print(f"     F1-Score : {best_f1*100:.2f}%")
print(f"     Accuracy : {best_row['Accuracy']*100:.2f}%")
print(f"     Recall   : {best_row['Recall']*100:.2f}%")
print(f"     Precision: {best_row['Precision']*100:.2f}%")
print(f"{'='*55}")


## 7. Visualisasi Hasil Benchmarking <a id='7'></a>

In [ ]:
# ── 7.1 Bar chart ROC-AUC semua model ────────────────────────────────────────
valid_df = results_df.dropna(subset=['ROC-AUC']).sort_values('ROC-AUC')

fig, ax = plt.subplots(figsize=(12, 8))
colors  = ['#E74C3C' if m == best_name else '#3498DB' for m in valid_df['Model']]

bars = ax.barh(valid_df['Model'], valid_df['ROC-AUC'] * 100,
               color=colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, valid_df['ROC-AUC'] * 100):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}%", va='center', fontsize=9.5,
            fontweight='bold' if val == valid_df['ROC-AUC'].max()*100 else 'normal')

ax.axvline(50, color='gray', linestyle='--', lw=1, label='Random (50%)')
ax.set_xlabel("ROC-AUC Score (%)", fontsize=12)
ax.set_title("📊 Leaderboard ROC-AUC — Semua Model", fontsize=14, pad=12)
ax.set_xlim(40, 105)
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("benchmark_roc_auc.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7.2 Scatter: ROC-AUC vs F1-Score ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    valid_df['ROC-AUC'] * 100,
    valid_df['F1-Score'] * 100,
    c=valid_df['Durasi(s)'],
    cmap='YlOrRd', s=100, edgecolors='gray', linewidths=0.5, zorder=3
)
plt.colorbar(scatter, ax=ax, label='Durasi Training (detik)')

for _, row in valid_df.iterrows():
    ax.annotate(
        row['Model'],
        (row['ROC-AUC']*100, row['F1-Score']*100),
        textcoords="offset points", xytext=(5, 4),
        fontsize=7.5, alpha=0.85
    )

# Highlight pemenang
ax.scatter(best_row['ROC-AUC']*100, best_row['F1-Score']*100,
           s=250, color='#E74C3C', zorder=5, label=f'🏆 {best_name}',
           edgecolors='darkred', linewidths=1.5)

ax.set_xlabel("ROC-AUC (%)", fontsize=12)
ax.set_ylabel("F1-Score (%)", fontsize=12)
ax.set_title("ROC-AUC vs F1-Score (warna = durasi training)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig("benchmark_scatter.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7.3 Radar chart 5 metrik untuk Top-5 model ───────────────────────────────
top5 = results_df.dropna(subset=['ROC-AUC']).head(5)
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
N = len(metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # tutup lingkaran

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
palette = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6']

for i, (_, row) in enumerate(top5.iterrows()):
    vals  = [row[m] for m in metrics]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2,
            color=palette[i], label=row['Model'])
    ax.fill(angles, vals, alpha=0.08, color=palette[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['20%','40%','60%','80%','100%'], fontsize=8)
ax.set_title("Radar Chart — Top 5 Model (5 Metrik)", fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
plt.tight_layout()
plt.savefig("benchmark_radar.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7.4 Confusion Matrix & ROC Curve model terbaik ───────────────────────────
best_pipeline = trained_pipelines[best_name]
y_pred_best   = best_pipeline.predict(X_test)

if hasattr(best_pipeline.named_steps['classifier'], 'predict_proba'):
    y_scores_best = best_pipeline.predict_proba(X_test)[:, 1]
elif hasattr(best_pipeline.named_steps['classifier'], 'decision_function'):
    y_scores_best = best_pipeline.decision_function(X_test)
else:
    y_scores_best = None

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Prediksi Lancar', 'Prediksi Default'],
            yticklabels=['Aktual Lancar',   'Aktual Default'],
            linewidths=0.5, linecolor='white')
tn, fp, fn, tp_val = cm.ravel()
axes[0].set_title(f"Confusion Matrix — {best_name}\n"
                  f"TP={tp_val:,}  TN={tn:,}  FP={fp:,}  FN={fn:,}", fontsize=12)

# ROC Curve
if y_scores_best is not None:
    fpr_b, tpr_b, _ = roc_curve(y_test, y_scores_best)
    auc_b = roc_auc_score(y_test, y_scores_best)
    axes[1].plot(fpr_b, tpr_b, color='#E74C3C', lw=2.5,
                 label=f'{best_name} (AUC={auc_b:.3f})')
    axes[1].plot([0,1],[0,1], 'k--', lw=1, label='Random')
    axes[1].fill_between(fpr_b, tpr_b, alpha=0.08, color='#E74C3C')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve — Model Terbaik', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.4)
else:
    axes[1].text(0.5, 0.5, 'ROC Curve tidak tersedia\n(model tidak support predict_proba)',
                 ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.savefig("best_model_eval.png", dpi=150, bbox_inches='tight')
plt.show()


## 8. Simpan & Load Model (Pickle) <a id='8'></a>

Pipeline yang disimpan ke pickle sudah **include preprocessor + classifier** dalam satu objek,
sehingga saat digunakan ulang tidak perlu scaling data secara manual.

| File | Isi |
|---|---|
| `BEST_<nama_model>.pkl` | Pipeline terbaik (preprocessor + classifier) |
| `all_models_benchmark.pkl` | Semua pipeline yang berhasil dilatih |
| `benchmark_results.csv` | Tabel hasil benchmarking |


In [ ]:
# ── 8.1 Simpan model terbaik ──────────────────────────────────────────────────
safe_name    = best_name.replace(" ", "_").replace("(","").replace(")","").strip()
best_pkl_path = os.path.join(MODEL_DIR, f"BEST_{safe_name}.pkl")

with open(best_pkl_path, 'wb') as f:
    pickle.dump(best_pipeline, f)

file_size = os.path.getsize(best_pkl_path) / 1024
print(f"✅ Model terbaik disimpan: {best_pkl_path}  ({file_size:.1f} KB)")


In [ ]:
# ── 8.2 Simpan semua pipeline terlatih ───────────────────────────────────────
all_pkl_path = os.path.join(MODEL_DIR, "all_models_benchmark.pkl")

with open(all_pkl_path, 'wb') as f:
    pickle.dump(trained_pipelines, f)

all_size = os.path.getsize(all_pkl_path) / (1024 * 1024)
print(f"✅ Semua model tersimpan: {all_pkl_path}  ({all_size:.1f} MB)")
print(f"   Total pipeline tersimpan: {len(trained_pipelines)}")


In [ ]:
# ── 8.3 Simpan tabel hasil benchmarking sebagai CSV ──────────────────────────
csv_path = "benchmark_results.csv"
results_df.to_csv(csv_path, index=False)
print(f"✅ Tabel hasil disimpan: {csv_path}")


In [ ]:
# ── 8.4 Verifikasi load ulang model terbaik ───────────────────────────────────
with open(best_pkl_path, 'rb') as f:
    loaded_pipeline = pickle.load(f)

y_pred_loaded  = loaded_pipeline.predict(X_test)
y_proba_loaded = None
if hasattr(loaded_pipeline.named_steps['classifier'], 'predict_proba'):
    y_proba_loaded = loaded_pipeline.predict_proba(X_test)[:, 1]

auc_loaded = roc_auc_score(y_test, y_proba_loaded) if y_proba_loaded is not None else np.nan

print(f"✅ Load ulang berhasil!")
print(f"   Tipe       : {type(loaded_pipeline).__name__}")
print(f"   Classifier : {type(loaded_pipeline.named_steps['classifier']).__name__}")
print(f"   ROC-AUC    : {auc_loaded:.4f}  (harus sama dengan sebelumnya: {best_auc:.4f})")
assert abs(auc_loaded - best_auc) < 1e-9, "❌ Model tidak cocok!"
print("   ✅ Identik — pickle tersimpan dengan benar")


In [ ]:
# ── 8.5 Download file pickle ke komputer (Google Colab) ───────────────────────
from google.colab import files

print("⬇️  Mengunduh file hasil benchmarking...")
files.download(best_pkl_path)
files.download(csv_path)
print("✅ Download selesai!")


## 9. Prediksi Nasabah Baru <a id='9'></a>

Fungsi `prediksi_nasabah()` menerima data nasabah baru dan menggunakan
model terbaik untuk memprediksi kemungkinan gagal bayar.
Karena menggunakan Pipeline, **tidak perlu scaling manual** — cukup masukkan nilai asli.


In [ ]:
def prediksi_nasabah(pipeline, nasabah_data: dict, threshold: float = 0.5):
    """
    Prediksi risiko gagal bayar untuk satu nasabah baru.

    Parameters
    ----------
    pipeline      : Pipeline sklearn yang sudah dilatih
    nasabah_data  : dict dengan semua kolom fitur (tanpa ID dan target)
    threshold     : batas probabilitas untuk klasifikasi (default 0.5)
    """
    input_df = pd.DataFrame([nasabah_data])

    prediksi = pipeline.predict(input_df)[0]

    prob = None
    if hasattr(pipeline.named_steps['classifier'], 'predict_proba'):
        prob = pipeline.predict_proba(input_df)[0][1]
    elif hasattr(pipeline.named_steps['classifier'], 'decision_function'):
        score = pipeline.decision_function(input_df)[0]
        prob  = 1 / (1 + np.exp(-score))   # sigmoid approximation

    # Override prediksi jika pakai threshold custom
    if prob is not None:
        prediksi = int(prob >= threshold)

    label = "🚨 GAGAL BAYAR (DEFAULT)" if prediksi == 1 else "✅ LANCAR (TIDAK DEFAULT)"
    warna = "\033[91m" if prediksi == 1 else "\033[92m"   # ANSI color
    reset = "\033[0m"

    print(f"{'='*52}")
    print(f"  HASIL PREDIKSI NASABAH")
    print(f"{'='*52}")
    print(f"  LIMIT_BAL  : {nasabah_data.get('LIMIT_BAL', '-'):>10,.0f}")
    print(f"  AGE        : {nasabah_data.get('AGE', '-'):>10}")
    print(f"  SEX        : {nasabah_data.get('SEX', '-'):>10}")
    print(f"  EDUCATION  : {nasabah_data.get('EDUCATION', '-'):>10}")
    print(f"  MARRIAGE   : {nasabah_data.get('MARRIAGE', '-'):>10}")
    print(f"  PAY_0      : {nasabah_data.get('PAY_0', '-'):>10}")
    print(f"{'='*52}")
    if prob is not None:
        bar_len = int(prob * 30)
        bar = "█" * bar_len + "░" * (30 - bar_len)
        print(f"  Prob. Default : [{bar}] {prob*100:.1f}%")
        print(f"  Threshold     : {threshold}")
    print(f"  Status     : {warna}{label}{reset}")
    print(f"{'='*52}")
    return prediksi, prob


print("✅ Fungsi prediksi_nasabah() siap digunakan")


In [ ]:
# ── Contoh 1: Nasabah Berisiko Rendah ─────────────────────────────────────────
print(">>> CONTOH 1: Nasabah dengan riwayat pembayaran baik\n")

nasabah_lancar = {
    'LIMIT_BAL': 200_000, 'SEX': 2, 'EDUCATION': 2, 'MARRIAGE': 1, 'AGE': 35,
    'PAY_0': -1, 'PAY_2': -1, 'PAY_3': -1, 'PAY_4': -1, 'PAY_5': -1, 'PAY_6': -1,
    'BILL_AMT1': 20_000, 'BILL_AMT2': 19_000, 'BILL_AMT3': 18_000,
    'BILL_AMT4': 17_000, 'BILL_AMT5': 16_000, 'BILL_AMT6': 15_000,
    'PAY_AMT1': 5_000,   'PAY_AMT2': 5_000,   'PAY_AMT3': 5_000,
    'PAY_AMT4': 5_000,   'PAY_AMT5': 5_000,   'PAY_AMT6': 5_000,
}

prediksi_nasabah(loaded_pipeline, nasabah_lancar)


In [ ]:
# ── Contoh 2: Nasabah Berisiko Tinggi ─────────────────────────────────────────
print(">>> CONTOH 2: Nasabah dengan riwayat keterlambatan berulang\n")

nasabah_default = {
    'LIMIT_BAL': 30_000,  'SEX': 1, 'EDUCATION': 3, 'MARRIAGE': 2, 'AGE': 28,
    'PAY_0': 3,  'PAY_2': 3, 'PAY_3': 2, 'PAY_4': 2, 'PAY_5': 1, 'PAY_6': 1,
    'BILL_AMT1': 29_000, 'BILL_AMT2': 28_500, 'BILL_AMT3': 28_000,
    'BILL_AMT4': 27_500, 'BILL_AMT5': 27_000, 'BILL_AMT6': 26_500,
    'PAY_AMT1': 0,       'PAY_AMT2': 0,       'PAY_AMT3': 500,
    'PAY_AMT4': 500,     'PAY_AMT5': 500,     'PAY_AMT6': 500,
}

prediksi_nasabah(loaded_pipeline, nasabah_default)


In [ ]:
# ── Contoh 3: Input interaktif (isi manual) ───────────────────────────────────
print(">>> CONTOH 3: Prediksi manual — ubah nilai di sini\n")

nasabah_custom = {
    # ── Data Pribadi ──────────────────────────────
    'LIMIT_BAL' : 50_000,   # Limit kredit (NT Dollar)
    'SEX'       : 1,         # 1=Laki-laki, 2=Perempuan
    'EDUCATION' : 2,         # 1=S2/S3, 2=S1, 3=SMA, 4=Lainnya
    'MARRIAGE'  : 2,         # 1=Menikah, 2=Lajang, 3=Lainnya
    'AGE'       : 30,        # Usia

    # ── Status Pembayaran (-1=tepat waktu, 1=telat 1bln, dst) ──
    'PAY_0': 0, 'PAY_2': 0, 'PAY_3': 0,
    'PAY_4': 0, 'PAY_5': 0, 'PAY_6': 0,

    # ── Tagihan Bulanan ───────────────────────────
    'BILL_AMT1': 10_000, 'BILL_AMT2': 9_500, 'BILL_AMT3': 9_000,
    'BILL_AMT4': 8_500,  'BILL_AMT5': 8_000, 'BILL_AMT6': 7_500,

    # ── Pembayaran Bulanan ────────────────────────
    'PAY_AMT1': 2_000, 'PAY_AMT2': 2_000, 'PAY_AMT3': 2_000,
    'PAY_AMT4': 2_000, 'PAY_AMT5': 2_000, 'PAY_AMT6': 2_000,
}

prediksi_nasabah(loaded_pipeline, nasabah_custom)


## 10. Kesimpulan <a id='10'></a>

In [ ]:
# ── Ringkasan akhir ───────────────────────────────────────────────────────────
valid_results = results_df.dropna(subset=['ROC-AUC'])
failed_results = results_df[results_df['ROC-AUC'].isna()]

print("=" * 60)
print("         RINGKASAN EKSPERIMEN BENCHMARKING")
print("=" * 60)
print(f"  Dataset            : UCI Credit Card Default")
print(f"  Total Sampel       : {len(df):,}")
print(f"  Rasio Default      : {df[TARGET].mean()*100:.1f}%")
print(f"  Ukuran Train/Test  : {1-TEST_SIZE:.0%} / {TEST_SIZE:.0%}")
print(f"  Total Model Diuji  : {len(all_models)}")
print(f"  Model Berhasil     : {len(valid_results)}")
print(f"  Model Gagal        : {len(failed_results)}")
print()
print(f"  ── Top 5 Model (ROC-AUC) ──────────────────────")
for _, row in valid_results.head(5).iterrows():
    print(f"  {int(row['Rank'])}.  {row['Model']:<28} AUC={row['ROC-AUC']*100:.2f}%  F1={row['F1-Score']*100:.2f}%")
print()
print(f"  🏆 PEMENANG : {best_name}")
print(f"     ROC-AUC  : {best_auc*100:.2f}%")
print(f"     F1-Score : {best_f1*100:.2f}%")
print("=" * 60)


### Temuan Utama

1. **Class Imbalance** (~22% default vs 78% lancar) ditangani dengan parameter `class_weight='balanced'` atau `is_unbalance=True` pada model yang mendukungnya.

2. **Fitur Prediktor Terkuat** berdasarkan analisis korelasi adalah kolom **PAY_0–PAY_6** (riwayat keterlambatan pembayaran) — konsisten dengan intuisi bahwa nasabah yang sering terlambat cenderung gagal bayar.

3. **Pipeline sklearn** memastikan preprocessing (StandardScaler) dan model menjadi satu unit yang bisa disimpan dan digunakan ulang via pickle tanpa risiko *data leakage*.

4. **ROC-AUC** digunakan sebagai metrik utama karena lebih robust terhadap class imbalance dibandingkan accuracy semata.

### Saran Pengembangan

- Lakukan **Hyperparameter Tuning** pada model terbaik menggunakan `GridSearchCV` atau `Optuna`
- Tambahkan **Feature Engineering**: rasio tagihan/limit, tren pembayaran, jumlah bulan terlambat berturut-turut
- Coba **Stacking / Voting Ensemble** dari Top-3 model untuk performa lebih tinggi
- Evaluasi dengan **threshold optimization** — turunkan threshold jika prioritas adalah *menangkap sebanyak mungkin default* (high recall)
